# 1.1 — Raw Data Profiling and Quality Control

Profile all subjects/trials in the raw dataset and run quality checks.

| | |
|---|---|
| **Inputs** | `data/raw/` (all subjects and trials) |
| **Outputs** | (visualization and summary tables only) |

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "1-experimentation" / "data" / "raw"

from gait_processing import load_trial, check_trial_quality

## 1. Discover available subjects and trials

In [ ]:
subject_dirs = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir()])

print(f"Found {len(subject_dirs)} subjects:")
for sd in subject_dirs:
    trials = sorted([t.name for t in sd.iterdir() if t.is_dir()])
    print(f"  {sd.name}: {len(trials)} trials — {trials}")

## 2. Profile each trial

In [ ]:
profiles = []

for subject_dir in subject_dirs:
    subject_id = subject_dir.name
    for trial_dir in sorted(subject_dir.iterdir()):
        if not trial_dir.is_dir():
            continue
        trial_id = trial_dir.name
        try:
            trial = load_trial(DATA_ROOT, subject_id, trial_id)
            profiles.append({
                "subject_id": subject_id,
                "trial_id": trial_id,
                "cohort": trial.cohort,
                "duration_s": trial.duration_s,
                "mass_kg": trial.mass_kg,
                "height_m": trial.height_m,
                "kinematic_rows": len(trial.trajectories),
                "analog_rows": len(trial.analog),
                "kinematic_cols": len(trial.trajectories.columns),
                "analog_cols": len(trial.analog.columns),
            })
        except Exception as e:
            profiles.append({
                "subject_id": subject_id,
                "trial_id": trial_id,
                "error": str(e),
            })

profile_df = pd.DataFrame(profiles)
profile_df

## 3. Run quality checks across all trials

In [ ]:
quality_results = []

for subject_dir in subject_dirs:
    subject_id = subject_dir.name
    for trial_dir in sorted(subject_dir.iterdir()):
        if not trial_dir.is_dir():
            continue
        trial_id = trial_dir.name
        try:
            trial = load_trial(DATA_ROOT, subject_id, trial_id)
            q = check_trial_quality(trial)
            quality_results.append({
                "subject_id": subject_id,
                "trial_id": trial_id,
                "kin_dup_frames": q.kinematic_integrity.duplicate_frames,
                "kin_non_mono": q.kinematic_integrity.non_monotonic_time,
                "kin_sampling_ok": q.kinematic_integrity.sampling_ok,
                "analog_dup_frames": q.analog_integrity.duplicate_frames,
                "analog_non_mono": q.analog_integrity.non_monotonic_time,
                "analog_sampling_ok": q.analog_integrity.sampling_ok,
            })
        except Exception as e:
            quality_results.append({
                "subject_id": subject_id,
                "trial_id": trial_id,
                "error": str(e),
            })

quality_df = pd.DataFrame(quality_results)
quality_df

In [ ]:
if "error" not in quality_df.columns:
    print("All trials passed quality checks.")
    print(f"  Kinematic sampling OK: {quality_df['kin_sampling_ok'].all()}")
    print(f"  Analog sampling OK: {quality_df['analog_sampling_ok'].all()}")
    print(f"  No duplicate frames: {(quality_df['kin_dup_frames'] == 0).all()}")
else:
    print("Some trials had errors:")
    display(quality_df[quality_df["error"].notna()])

## Summary

All subjects and trials are profiled. Quality checks validate timing integrity and signal completeness.

**Next step:** `1_2_data_raw_preprocessing.ipynb` — clean signals, detect gait events, and build normalized gait cycles.